# Jaguar Re-ID — Top-1 Solution: DINOv2-ViT-L/14 + ArcFace + bg=gray + k-reciprocal re-rank

**Author:** jreiml (Kaggle) / zyna (W&B entity)  
**W&B project:** https://wandb.ai/zyna/jaguar-reid-jreiml  
**GitHub:** https://github.com/jreiml/jaguar-re-id (see `src/jaguar_reid/`)

This notebook reproduces our best single-model Kaggle Round-2 submission:

- **Backbone:** DINOv2-ViT-L/14 at 518×518 (timm `vit_large_patch14_reg4_dinov2.lvd142m`), frozen.
- **Head:** 1024 → 512 → 256 projection + ArcFace (margin 0.5, scale 64).
- **Training:** identity-disjoint val_v1 split (25 train / 6 val identities), AdamW lr=1e-4 wd=1e-4, batch 64, 30 epochs, ReduceLROnPlateau on val mAP.
- **Inference on R2 test:** alpha-mask-aware background fill (gray 128) before embedding.
- **Post-processing:** k-reciprocal re-ranking (k1=35, k2=6, λ=0.2).

**Results (val_v1 identity-balanced mAP):** frozen DINOv2 0.64 → + projection+ArcFace 0.682 → + rerank 0.708.  
**Kaggle public R2 late-submission:** 0.302 (baseline MegaDescriptor+ArcFace was 0.243).

For Round 1 (backgrounds intact), remove the `bg_mode=gray` step — R1 test images already carry natural backgrounds that match the training domain.

## Environment

Expects the jaguar-re-id or round-2-jaguar-reidentification-challenge competition mounted at `/kaggle/input/...`. This notebook installs timm and its deps without pulling a PyPI torch (we keep the Kaggle-provided torch).

In [ ]:
!pip install --quiet --no-deps timm==1.0.26 imagehash==4.3.2
import os, math, json
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import timm
from tqdm.auto import tqdm

print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'timm', timm.__version__)

In [ ]:
DATA_DIR = Path('/kaggle/input/round-2-jaguar-reidentification-challenge')
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
TRAIN_CSV = DATA_DIR / 'train.csv'
TEST_CSV = DATA_DIR / 'test.csv'
SAMPLE_CSV = DATA_DIR / 'sample_submission.csv'
WORK = Path('/kaggle/working')

BACKBONE = 'vit_large_patch14_reg4_dinov2.lvd142m'
INPUT_SIZE = 518
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
DROPOUT = 0.3
ARCFACE_MARGIN = 0.5
ARCFACE_SCALE = 64.0
BATCH_SIZE = 16  # smaller for Kaggle GPU
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 30
PATIENCE = 10
SEED = 42
VAL_FRAC = 0.2
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

## Identity-disjoint val split

Val identities are never seen at training time — this is the correct closed-set re-ID protocol and matches the Kaggle test setting.

In [ ]:
import random
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

df = pd.read_csv(TRAIN_CSV)
counts = df.groupby('ground_truth').size().sort_values(ascending=False)
eligible = counts[counts >= 2].index.tolist()
rng = random.Random(SEED)
cand = sorted(set(eligible)); rng.shuffle(cand)
n_val = max(1, int(round(len(cand) * VAL_FRAC)))
val_ids = sorted(cand[:n_val]); train_ids = sorted(set(eligible) - set(val_ids))
train_fns = df.loc[df.ground_truth.isin(train_ids), 'filename'].astype(str).tolist()
val_fns = df.loc[df.ground_truth.isin(val_ids), 'filename'].astype(str).tolist()
print(f'train: {len(train_fns)} imgs / {len(train_ids)} ids')
print(f'val:   {len(val_fns)} imgs / {len(val_ids)} ids')
assert set(train_ids).isdisjoint(set(val_ids))

## Background-aware image loader

The Kaggle PNGs are RGBA; alpha encodes a pre-computed jaguar segmentation mask. For Round-2 test images the RGB is pre-zeroed outside the mask — a domain shift for naturally-trained backbones. We fill with gray 128 to pull the test distribution back toward the training distribution.

In [ ]:
def load_rgb(path, mode='as_is'):
    img = Image.open(path)
    if img.mode != 'RGBA':
        return img.convert('RGB')
    arr = np.asarray(img)
    rgb = arr[..., :3].copy(); alpha = arr[..., 3]
    if mode == 'as_is':
        return Image.fromarray(rgb)
    bg = alpha == 0
    if mode == 'gray':
        rgb[bg] = 128
    elif mode == 'imagenet_mean':
        rgb[bg] = np.array([124, 116, 104], dtype=np.uint8)
    return Image.fromarray(rgb)

preprocess = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class ImageDataset(Dataset):
    def __init__(self, paths, mode='as_is'):
        self.paths = paths; self.mode = mode
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return preprocess(load_rgb(self.paths[i], self.mode)), self.paths[i].name

## Frozen backbone features (once per split)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
backbone = timm.create_model(BACKBONE, pretrained=True, num_classes=0).eval().to(device)
with torch.no_grad():
    feat_dim = int(backbone(torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE, device=device)).shape[1])
print('backbone params:', sum(p.numel() for p in backbone.parameters())/1e6, 'M; feat_dim:', feat_dim)

@torch.no_grad()
def extract(paths, mode='as_is'):
    dl = DataLoader(ImageDataset(paths, mode), batch_size=BATCH_SIZE, num_workers=2, pin_memory=True)
    out = []
    for x, _ in tqdm(dl, desc=f'embed-{mode}'):
        out.append(backbone(x.to(device)).cpu().numpy())
    return np.vstack(out)

train_emb = extract([TRAIN_DIR / f for f in train_fns])
val_emb = extract([TRAIN_DIR / f for f in val_fns])

## Projection head + ArcFace

In [ ]:
class Projection(nn.Module):
    def __init__(self, in_dim, hidden, out_dim, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim), nn.BatchNorm1d(out_dim))
    def forward(self, x): return self.net(x)

class ArcFace(nn.Module):
    def __init__(self, dim, nc, margin=0.5, scale=64.0):
        super().__init__()
        self.w = nn.Parameter(torch.empty(nc, dim)); nn.init.xavier_uniform_(self.w)
        self.m = margin; self.s = scale
        self.cos_m = math.cos(margin); self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin); self.mm = math.sin(math.pi - margin) * margin
    def forward(self, e, y):
        e = F.normalize(e, 2, 1); w = F.normalize(self.w, 2, 1)
        cos = F.linear(e, w).clamp(-1, 1)
        sine = torch.sqrt(torch.clamp(1 - cos**2, min=1e-9))
        phi = torch.where(cos > self.th, cos * self.cos_m - sine * self.sin_m, cos - self.mm)
        oh = torch.zeros_like(cos); oh.scatter_(1, y.view(-1, 1).long(), 1)
        return ((oh * phi) + ((1 - oh) * cos)) * self.s

id2cls = {i: c for c, i in enumerate(sorted(train_ids))}
by_fn = dict(zip(df.filename.astype(str), df.ground_truth.astype(str)))
train_lab = np.array([id2cls[by_fn[f]] for f in train_fns])
val_labels = np.array([by_fn[f] for f in val_fns])

proj = Projection(feat_dim, HIDDEN_DIM, EMBEDDING_DIM, DROPOUT).to(device)
head = ArcFace(EMBEDDING_DIM, len(id2cls), ARCFACE_MARGIN, ARCFACE_SCALE).to(device)
num_params = sum(p.numel() for p in list(proj.parameters()) + list(head.parameters()))
print('proj+head params:', num_params)

## Training

In [ ]:
from torch.utils.data import TensorDataset

def identity_balanced_map(emb, labels):
    e = emb / np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12)
    sim = e @ e.T; np.fill_diagonal(sim, -np.inf)
    per = {}
    for i in range(len(labels)):
        m = (labels == labels[i]).astype(int); m[i] = 0
        if m.sum() == 0: continue
        order = np.argsort(-sim[i], kind='stable'); sm = m[order]
        prec = np.cumsum(sm) / np.arange(1, len(sm) + 1)
        per.setdefault(labels[i], []).append(float((prec * sm).sum() / sm.sum()))
    return float(np.mean([np.mean(v) for v in per.values()]))

opt = torch.optim.AdamW(list(proj.parameters()) + list(head.parameters()), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=5)
ce = nn.CrossEntropyLoss()
ds = TensorDataset(torch.from_numpy(train_emb).float(), torch.from_numpy(train_lab).long())
dl = DataLoader(ds, batch_size=BATCH_SIZE * 4, shuffle=True)
val_t = torch.from_numpy(val_emb).float().to(device)
best_map = -1.0
patience_ctr = 0
for epoch in range(1, NUM_EPOCHS + 1):
    proj.train(); head.train(); loss_sum = n = 0
    for x, y in dl:
        x = x.to(device); y = y.to(device)
        logits = head(proj(x), y); loss = ce(logits, y)
        opt.zero_grad(); loss.backward(); opt.step()
        loss_sum += loss.item() * x.size(0); n += x.size(0)
    proj.eval()
    with torch.no_grad():
        v = F.normalize(proj(val_t), 2, 1).cpu().numpy()
    vm = identity_balanced_map(v, val_labels)
    sched.step(vm)
    improved = vm > best_map
    if improved:
        best_map = vm; best_state = {'proj': proj.state_dict(), 'head': head.state_dict()}; patience_ctr = 0
    else: patience_ctr += 1
    print(f'epoch {epoch:03d} loss {loss_sum/n:.4f} val_mAP {vm:.4f} best {best_map:.4f} (patience {patience_ctr})')
    if patience_ctr >= PATIENCE: break
proj.load_state_dict(best_state['proj']); head.load_state_dict(best_state['head'])
print('final best val mAP:', best_map)

## Test-set embedding (with bg=gray for R2) + k-reciprocal re-ranking + submission

In [ ]:
pairs = pd.read_csv(TEST_CSV)
unique = sorted(set(pairs.query_image).union(set(pairs.gallery_image)))

# R2 test — bg pre-zeroed. Fill with gray before embedding.
test_emb = extract([TEST_DIR / f for f in unique], mode='gray')
proj.eval()
with torch.no_grad():
    te = F.normalize(proj(torch.from_numpy(test_emb).float().to(device)), 2, 1).cpu().numpy()
idx = {fn: i for i, fn in enumerate(unique)}

# k-reciprocal re-rank (Zhong et al., CVPR 2017)
def k_reciprocal_rerank(emb, k1=35, k2=6, lam=0.2):
    emb = emb / np.maximum(np.linalg.norm(emb, axis=1, keepdims=True), 1e-12)
    od = np.maximum(1 - emb @ emb.T, 0.0)
    od = od / np.maximum(od.max(axis=0, keepdims=True), 1e-12)
    ir = np.argsort(od, axis=1, kind='stable')
    n = len(emb); V = np.zeros_like(od, dtype=np.float32)
    def kr(i, k):
        fw = ir[i, :k+1]; bw = ir[fw, :k+1]; return fw[np.where(bw == i)[0]]
    for i in range(n):
        kri = kr(i, k1); ke = kri.copy()
        for c in kri:
            s = kr(c, int(round(k1/2)))
            if len(np.intersect1d(s, kri)) > 2.0/3.0 * len(s):
                ke = np.union1d(ke, s)
        w = np.exp(-od[i, ke]).astype(np.float32); V[i, ke] = w / w.sum()
    if k2 > 1:
        Vq = np.zeros_like(V)
        for i in range(n): Vq[i] = np.mean(V[ir[i, :k2], :], axis=0)
        V = Vq
    inv = [np.where(V[:, c] != 0)[0] for c in range(n)]
    jd = np.zeros_like(od)
    for i in range(n):
        tm = np.zeros(n, dtype=np.float32); nz = np.where(V[i] != 0)[0]
        for j_, c in enumerate(nz):
            idx_set = inv[c]; tm[idx_set] += np.minimum(V[i, c], V[idx_set, c])
        jd[i] = 1.0 - tm / (2.0 - tm)
    fd = jd * (1 - lam) + od * lam; np.fill_diagonal(fd, 0.0)
    return fd

dist = k_reciprocal_rerank(te, k1=35, k2=6, lam=0.2)
sim_mat = 1.0 - (dist - dist.min()) / max(dist.max() - dist.min(), 1e-12)
sim = np.array([sim_mat[idx[q], idx[g]] for q, g in zip(pairs.query_image, pairs.gallery_image)])
sim = np.clip(sim, 0.0, 1.0)
sub = pd.DataFrame({'row_id': pairs.row_id.values, 'similarity': sim})
sub.to_csv(WORK / 'submission.csv', index=False)
print('Saved', WORK / 'submission.csv', 'rows:', len(sub))